# 01 - Data Preparation
Parsing the periapical lesion dataset XML annotations, building manifests, and creating train/val/test splits for classification, detection, and segmentation.


In [ ]:
import sys, os
# Find project root (works from project root or notebooks/ subdir)
_root = None
for _p in [os.path.abspath('.'), os.path.abspath('..')]:
    if os.path.exists(os.path.join(_p, 'src', 'config.py')):
        _root = _p
        break
if _root:
    sys.path.append(_root)
else:
    raise RuntimeError('Could not find project root (src/config.py not found)')
from src.data_utils import *
from src.config import *
import random
random.seed(RANDOM_SEED)


In [ ]:
# Scan dataset directories
original_dir, aug_dir, annot_dir = scan_dataset()
if original_dir is None or annot_dir is None:
    raise FileNotFoundError(f'Could not find dataset in {DATA_RAW}. '
                              'Extract the Mendeley download into data/raw/.')
print(f'Original images dir: {original_dir}')
print(f'Annotation dir: {annot_dir}')


In [ ]:
# Build manifest from original images only (not augmented)
manifest = build_dataset_manifest(original_dir, annot_dir, sample_size=NUM_SAMPLES)
print(f'Total samples in manifest: {len(manifest)}')


In [ ]:
# Check class distribution
dist = get_class_distribution(manifest)
print('Class distribution:', dist)


In [ ]:
# Split into train/val/test
splits = split_dataset(manifest)
for k, v in splits.items():
    print(f'{k}: {len(v)} samples')


## Prepare Classification Data


In [ ]:
cls_records = prepare_classification_data(splits)
print(f'Classification records: {len(cls_records)}')


## Prepare Detection Data (YOLO format)


In [ ]:
det_records, yaml_path = prepare_detection_data(splits)
print(f'Detection records: {len(det_records)}')
print(f'YAML config: {yaml_path}')


## Prepare Segmentation Data (Box-to-Mask conversion)


In [ ]:
seg_records = prepare_segmentation_data(splits)
print(f'Segmentation records: {len(seg_records)}')


In [ ]:
# Save split indices for later reuse
import pickle
split_info = {k: [m["filename"] for m in v] for k, v in splits.items()}
with open(os.path.join(DATA_PROCESSED, 'splits.pkl'), 'wb') as f:
    pickle.dump(split_info, f)
print('Data preparation complete!')
